In [0]:
from pyspark.sql import functions as F


In [0]:
client_contracts_clean = spark.table("client_contracts_clean")
client_monthly_model_base = spark.table("client_monthly_model_base")

prediction_date = contract_end_date - 3 months

In [0]:
# Create the "as-of" data when the model should score the account. 
# If the contract end is in February 2024, I pretend I am standing in November 2023 and ask "Will this client renew?"
prediction_snapshot = (
    client_contracts_clean
    .withColumn(
        "prediction_date", 
        F.add_months(F.col("contract_end_date"), -3)
    )
    .withColumn(
        "prediction_month",
        F.date_trunc("month", F.col("prediction_date"))
    )
)

display(prediction_snapshot)

In [0]:
# Keep only the variables that I need 
prediction_snapshot = (
    prediction_snapshot
    .select(
        "client_id",
        "client_name",
        "industry",
        "region",
        "company_size",
        "eligible_employees",
        "annual_contract_value",
        "contract_start_date",
        "contract_end_date",
        "contract_term_months",
        "renewal_cycle_id",
        "renewed_flag",
        "non_renewal_flag",
        "prediction_date",
        "prediction_month"
    )
)

display(prediction_snapshot)

In [0]:
# Validate one row per renewal_cycle_id 
display(
    prediction_snapshot
    .groupBy("renewal_cycle_id")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
# Check that the prediction month exists in the monthly base to confirm that for each client, there is a row in the client_monthly_model_base for the prediction month 
prediction_month_check = (
    prediction_snapshot.alias("p")
    .join(
        client_monthly_model_base.select("client_id","month").alias("m"),
        (F.col("p.client_id") == F.col("m.client_id")) & (F.col("p.prediction_month") == F.col("m.month")),
        how="left"
        )
    .withColumn(
        "prediction_month_exists",
        F.when(F.col("m.month").isNotNull(),1).otherwise(0)
        )
    )

display(prediction_month_check.groupBy("prediction_month_exists").count())

In [0]:
display(client_monthly_model_base
        .agg(F.min("month").alias("min_month"),
             F.max("month").alias("max_month"))
)

In [0]:
prediction_snapshot_valid = ( 
                             prediction_snapshot.alias("p")
                             .join(
                                 client_monthly_model_base.select("client_id","month").alias("m"),
                                 (F.col("p.client_id") == F.col("m.client_id")) &
                                 (F.col("p.prediction_month") == F.col("m.month")),
                                 how = "inner"
                             )
                             .select("p.*")
)

In [0]:
display(
    prediction_snapshot
    .filter(F.col("prediction_month") > F.lit("2024-12-01"))
    .select("client_id", "contract_end_date", "prediction_month")
    .orderBy("prediction_month")
)

In [0]:
# Build a strict snapshot table with the actual monthly row at T-3 to get the exact monthly record corresponding to the prediction month.
prediction_snapshot_base = (
    prediction_snapshot_valid.alias("p")
    .join(
        client_monthly_model_base.alias("m"),
        (F.col("p.client_id") == F.col("m.client_id")) &
        (F.col("p.prediction_month") == F.col("m.month")),
        how="left"
    )
    .select(
        F.col("p.client_id"),
        F.col("p.client_name"),
        F.col("p.industry"),
        F.col("p.region"),
        F.col("p.company_size"),
        F.col("p.eligible_employees"),
        F.col("p.annual_contract_value"),
        F.col("p.contract_start_date"),
        F.col("p.contract_end_date"),
        F.col("p.contract_term_months"),
        F.col("p.renewal_cycle_id"),
        F.col("p.renewed_flag"),
        F.col("p.non_renewal_flag"),
        F.col("p.prediction_date"),
        F.col("p.prediction_month"),
        F.col("m.months_to_renewal"),
        F.col("m.latent_risk_segment")
    )
)

display(prediction_snapshot_base)

In [0]:
print("prediction_snapshot rows:", prediction_snapshot_valid.count())
print("prediction_snapshot_base rows:", prediction_snapshot_base.count())

In [0]:
display(
    prediction_snapshot_valid
    .select(
        "client_id",
        "contract_end_date",
        "prediction_month"
    )
    .orderBy("contract_end_date")
)

In [0]:
display(
    prediction_snapshot_valid
    .groupBy("renewed_flag", "non_renewal_flag")
    .count()
)

In [0]:
spark.sql("DROP TABLE IF EXISTS prediction_snapshot_valid")
spark.sql("DROP TABLE IF EXISTS prediction_snapshot_base")

In [0]:
prediction_snapshot_valid.write.mode("overwrite").saveAsTable("prediction_snapshot_valid")
prediction_snapshot_base.write.mode("overwrite").saveAsTable("prediction_snapshot_base")